### Recuit Simulé 
_______________________________________________________________________________________________________________
### Auteur : Luc Carlos Asso

### _______________________________________________________________________________________________________________

### Importation des Packages

### _______________________________________________________________________________________________________________

In [ ]:
import re
import numpy as np
import random
import matplotlib.pyplot as plt
import time 

### _______________________________________________________________________________________________________________

### Recuit en mode classe

### """  Informations :
            ‣ read_instance(filename) : recupération des instances
            ‣ Fonction_objectif( x : np.ndarray) : Fonction Objectif
            ‣ Solution_Initiale() : Solution Intiale
            ‣ voisin_swp(x : np.ndarray): Voisin de x
            ‣ Sauvegarder(solution : np.ndarray) : Sauvegarder les solutions
            ‣ Meilleurs_solutions() : les meilleurs solutions réalisables
     """

In [ ]:
class recuit_simule :
    
    @staticmethod
    def read_instance(filename):
        with open(filename, "r") as f:
            lines = [line.strip() for line in f if line.strip()]
        n = int(re.findall(r'\d+', lines[1])[0])                                      # -------- n --------------
        profits = [[0]*n for _ in range(n)]                                           # -------- profits --------
        idx = 3
        for i in range(n):
            nums = list(map(int, re.findall(r'-?\d+', lines[idx])))
            for j in range(i+1):
                profits[i][j] = nums[j]
                profits[j][i] = nums[j]
            idx += 1

        idx += 1  # saute "#Weights:"
        weights = list(map(int, re.findall(r'-?\d+', lines[idx])))                    # -------- poids -----------
        capacity = int(re.findall(r'\d+', lines[idx+1])[0])                           # -------- capacity --------
        k = int(re.findall(r'\d+', lines[idx+2])[0])                                  # -------- k ---cardinalité-
        return n, profits, weights, capacity, k

    """                                                                               #  A décocher après la prémière exécution  
    n, P, W, C, k = recuit_simule.read_instance("Instance_180_100_100_9.txt")
    P = np.array(P)
    W = np.array(W)  
    """
    @staticmethod
    def Fonction_objectif( x : np.ndarray) :
        Val1 = 0
        for i in range(recuit_simule.n) :
            Val1+=recuit_simule.P[i,i]*x[i]*x[i]
            for j in range(i+1,recuit_simule.n) : 
                    Val1+= recuit_simule.P[i,j]*x[i]*x[j]
        return Val1
        
 
    
    @staticmethod
    def Solution_Initiale() : 
        dict_test = {}
        dict_stock = {}
        nb_objet= recuit_simule.n
        W_T = sum(recuit_simule.W)
        Sol_I = [1]*recuit_simule.n
        profits = np.sum(recuit_simule.P, axis=1)
        for i in range(recuit_simule.n) : 
            dict_test[i] = profits[i]/recuit_simule.W[i]
        for i in range(recuit_simule.n - recuit_simule.k) : 
            ind = sorted(dict_test, key=dict_test.get)[0]
            Sol_I[ind]=0
            W_T-= recuit_simule.W[ind]
            nb_objet-=1
            dict_stock[ind]= dict_test[ind]
            del dict_test[ind]
            if W_T <= recuit_simule.C and nb_objet == recuit_simule.k :
                break
            while nb_objet == recuit_simule.k and W_T > recuit_simule.C : 
                ind = sorted(dict_test, key=dict_test.get)[0]
                j=0
                for i in list(dict_stock.keys())[::-1] :
                    if recuit_simule.W[ind] - recuit_simule.W[i] > 0 :
                        Sol_I[ind]=0
                        Sol_I[i]=1
                        W_T+= recuit_simule.W[i] - recuit_simule.W[ind]  
                        j-=1
                        dict_test[i]= profits[i]/recuit_simule.W[i]
                        del dict_test[ind]
                        break
                    j+=1
                if j == len(list(dict_stock.keys())) :
                    del dict_test[ind]
        return np.array(Sol_I)           
                               
    @staticmethod
    def voisin_swp(x: np.ndarray):
        v = x.copy()
        essais=0
        while  essais < 50 :
            indices_1 = [i for i in range(len(v)) if v[i] == 1]
            indices_0 = [i for i in range(len(v)) if v[i] == 0]
            i = random.choice(indices_1)
            j = random.choice(indices_0)
            v[i] = 0
            v[j] = 1
            if np.dot(v, recuit_simule.W) <= recuit_simule.C:
                break
            essais += 1
        return v
        
    """
        Paramètres 
            ‣ taux_d  taux de décroissance
            ‣ ɛ : paramètre de condition 
            ‣ R : nombre d'itération pour chaque pas de croissance
     """
   
    @staticmethod 
    def Recuit_simulé(taux_d : float, ɛ : float, R : int ):
        Sol_initiale = recuit_simule.Solution_Initiale()
        meilleur = Sol_initiale
        T = 2*recuit_simule.n           # T0
        D = 0                                                        # Palier
        DV = 0
        while T > ɛ :
            for i in range(R) :
                V_s = recuit_simule.voisin_swp(Sol_initiale)
                F_V_s = recuit_simule.Fonction_objectif(V_s)         # image de  V_s par Fonction_objectif
                F_s = recuit_simule.Fonction_objectif(Sol_initiale)  # image de la solution initiale par Fonction_objectif
                if F_V_s >  F_s : 
                    Sol_initiale = V_s
                    meilleur = V_s
                    D = 0
                    DV = 0 
                else:
                    if D == 10 or DV == 10  :
                        time.sleep(30)
                    r = random.random()
                    ΔF =  F_V_s -  F_s
                    s=np.exp(ΔF/T)
                    if r <= np.exp(ΔF/T) :
                        Sol_initiale = V_s 
                        DV+=1
                    else :
                        D+=1
            T=T*taux_d
        return meilleur

### Exécution 

### VALEURS DES PARAMETRES

In [ ]:
taux_d , ɛ , R = 0.75,0.0095,15

In [ ]:
debut= time.time()
Solution = recuit_simule.Recuit_simulé(taux_d , ɛ , R)
fin= time.time()
temps= fin - debut

### Temps d'exécution

In [ ]:
heures = int(temps // 3600)
minutes = int ((temps % 3600) // 60)
secondes =  temps % 60
print(f"Temps d'exécution : {heures} h {minutes} min {secondes} s")

### Solution réalisable approchée par le recuit simulé

In [ ]:
Solution[0]

### valeur par Fonction_objectif de la solution

In [ ]:
print(recuit_simule.Fonction_objectif(Solution[0]))

### Indices des 1 de la solution

In [ ]:
sol = Solution[0]
indices = [i+1 for i,v in enumerate(sol) if v == 1]
print(indices)

### _______________________________________________________________________________________________________________

### Exécuter une instance sur plusieurs paramètres mais à temps constant 

### _______________________________________________________________________________________________________________

In [ ]:
int_td = np.linspace(0.75,0.95,3)
int_ɛ = np.linspace(0.001,0.149,3)
Dict={}
T = 2*recuit_simule.n 
debut= time.time()
for i in sorted(int_td):
    for j in sorted(int_ɛ):
        sol = recuit_simule.Recuit_simulé(i,j,15)                        
        indices = [i+1 for i,v in enumerate(sol) if v == 1]
        Dict[(T,i,j,tuple(indices))]= recuit_simule.Fonction_objectif(sol) 
fin= time.time()
temps= fin - debut

In [ ]:
Dict

### Temps d'exécution

In [ ]:
heures = int(temps // 3600)
minutes = int ((temps % 3600) // 60)
secondes =  temps % 60
print(f"Temps d'exécution : {heures} h {minutes} min {secondes} s")

### A Exécuter avant d'utiliser une autre instance

In [ ]:
recuit_simule.n=0
recuit_simule.C=0
recuit_simule.P=[]
recuit_simule.k=0
recuit_simule.W=[]

### _______________________________________________________________________________________________________________

### _______________________________________________________________________________________________________________

### _______________________________________________________________________________________________________________

### _______________________________________________________________________________________________________________